In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# Load Dataset
housing = fetch_california_housing()

X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target)

In [3]:
# Basic Data Understanding
print(X.head())


   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  
0    -122.23  
1    -122.22  
2    -122.24  
3    -122.25  
4    -122.25  


In [4]:
print(X.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOccup    20640 non-null  float64
 6   Latitude    20640 non-null  float64
 7   Longitude   20640 non-null  float64
dtypes: float64(8)
memory usage: 1.3 MB
None


In [5]:
print(X.describe())

             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
mean       3.870671     28.639486      5.429000      1.096675   1425.476744   
std        1.899822     12.585558      2.474173      0.473911   1132.462122   
min        0.499900      1.000000      0.846154      0.333333      3.000000   
25%        2.563400     18.000000      4.440716      1.006079    787.000000   
50%        3.534800     29.000000      5.229129      1.048780   1166.000000   
75%        4.743250     37.000000      6.052381      1.099526   1725.000000   
max       15.000100     52.000000    141.909091     34.066667  35682.000000   

           AveOccup      Latitude     Longitude  
count  20640.000000  20640.000000  20640.000000  
mean       3.070655     35.631861   -119.569704  
std       10.386050      2.135952      2.003532  
min        0.692308     32.540000   -124.350000  
25%        2.429741     33.930000   -1

In [6]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
# Feature Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
# Train Model (UC-8)
model = LinearRegression()
model.fit(X_train_scaled, y_train)

LinearRegression()

In [9]:
# Evaluate Model
y_pred = model.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R2 Score:", r2)

RMSE: 0.7455813830127763
R2 Score: 0.575787706032451


In [10]:
# Residuals
residuals = y_test - y_pred

In [11]:
# Feature Names
feature_columns = X.columns

## HouseBroker Key Components

In [12]:
# InputProcessor (UC-2)
def preprocess_input(input_df, scaler, feature_columns):
    input_df = input_df[feature_columns]
    return scaler.transform(input_df)

In [13]:
# RegressionModel (UC-3)
def predict_price(model, processed_input):
    return model.predict(processed_input)[0]

In [14]:
# Price Interval (UC-3)
def compute_interval(pred, residuals):
    std = np.std(residuals)
    lower = pred - 1.96 * std
    upper = pred + 1.96 * std
    return lower, upper

In [15]:
# Classification (UC-4, UC-5)
def classify_price(pred, listing_price=None):
    if listing_price is None:
        return "No comparison"
    
    diff = (listing_price - pred) / pred * 100
    
    if diff > 10:
        return "Overpriced"
    elif diff < -10:
        return "Underpriced"
    else:
        return "Fairly priced"

In [16]:
# Feature Importance (UC-8)
def feature_importance(model, feature_columns):
    importance = dict(zip(feature_columns, model.coef_))
    return sorted(importance.items(), key=lambda x: abs(x[1]), reverse=True)

In [22]:
# classify_price (UC-4)

# Pricing Classification (UC-4)
def classify_price(predicted_price, listing_price):
    if listing_price is None:
        return "No Listing Provided"
    
    deviation = (listing_price - predicted_price) / predicted_price
    
    if deviation > 0.1:
        return "Overpriced"
    elif deviation < -0.1:
        return "Underpriced"
    else:
        return "Fairly Priced"

In [17]:
# Full System Pipeline
def housebroker_system(
    input_data,
    model,
    scaler,
    feature_columns,
    residuals,
    listing_price=None
):
    
    processed = preprocess_input(input_data, scaler, feature_columns)
    
    pred = predict_price(model, processed)
    
    lower, upper = compute_interval(pred, residuals)
    
    classification = classify_price(pred, listing_price)
    
    important_features = feature_importance(model, feature_columns)[:5]
    
    return {
        "Predicted Price": pred,
        "Price Range": (lower, upper),
        "Classification": classification,
        "Top Features": important_features
    }

In [18]:
# End User
sample_input = X_test.iloc[[0]]
listing_price = y_test.iloc[0] * 1.1

result = housebroker_system(
    sample_input,
    model,
    scaler,
    feature_columns,
    residuals,
    listing_price
)

result

{'Predicted Price': 0.7191228416019138,
 'Price Range': (-0.7422007574018947, 2.1804464406057225),
 'Classification': 'Underpriced',
 'Top Features': [('Latitude', -0.8969288766386657),
  ('Longitude', -0.8698417752417166),
  ('MedInc', 0.8543830309268546),
  ('AveBedrms', 0.3392594905944831),
  ('AveRooms', -0.2944101344732982)]}

In [19]:
# Format Output
print("----- HouseBroker Prediction -----")
print(f"Predicted Price: {result['Predicted Price']:.2f}")
print(f"Price Range: {result['Price Range']}")
print(f"Classification: {result['Classification']}")

print("\nTop Influential Features:")
for f, v in result["Top Features"]:
    print(f"{f}: {v:.4f}")

----- HouseBroker Prediction -----
Predicted Price: 0.72
Price Range: (-0.7422007574018947, 2.1804464406057225)
Classification: Underpriced

Top Influential Features:
Latitude: -0.8969
Longitude: -0.8698
MedInc: 0.8544
AveBedrms: 0.3393
AveRooms: -0.2944


In [23]:
# Unit Tests 

# Test 1: Input Preprocessing
def test_preprocess_input():
    sample = X_test.iloc[[0]]
    
    processed = preprocess_input(sample, scaler, feature_columns)
    
    assert processed.shape[1] == len(feature_columns)
    print("PASS: Input preprocessing works correctly")

In [25]:
# Test 2: Prediction Function
def test_prediction():
    sample = X_test.iloc[[0]]
    processed = preprocess_input(sample, scaler, feature_columns)
    
    pred = predict_price(model, processed)
    
    assert isinstance(pred, float)
    print("PASS: Prediction generated →", pred)

In [26]:
# Test 3: Interval Calculation
def test_interval():
    sample = X_test.iloc[[0]]
    processed = preprocess_input(sample, scaler, feature_columns)
    
    pred = predict_price(model, processed)
    lower, upper = compute_interval(pred, residuals)
    
    assert lower < pred < upper
    print("PASS: Interval calculation correct")

In [27]:
# Test 4: Classification Logic
def test_classification():
    pred = 2.0
    
    assert classify_price(pred, 2.5) == "Overpriced"
    assert classify_price(pred, 1.5) == "Underpriced"
    assert classify_price(pred, 2.05) == "Fairly Priced"
    
    print("PASS: Classification logic correct")

In [28]:
# Test 5: Feature Importance
def test_feature_importance():
    features = feature_importance(model, feature_columns)
    
    assert len(features) > 0
    assert isinstance(features[0], tuple)
    
    print("PASS: Feature importance computed")

In [29]:
# Test 6: Full Pipeline Test
def test_full_pipeline():
    sample = X_test.iloc[[0]]
    listing_price = y_test.iloc[0] * 1.1
    
    result = housebroker_system(
        sample,
        model,
        scaler,
        feature_columns,
        residuals,
        listing_price
    )
    
    assert "Predicted Price" in result
    assert "Price Range" in result
    assert "Classification" in result
    
    print("PASS: Full system pipeline works")

In [30]:
# Run All Tests
def run_all_tests():
    test_preprocess_input()
    test_prediction()
    test_interval()
    test_classification()
    test_feature_importance()
    test_full_pipeline()

run_all_tests()

PASS: Input preprocessing works correctly
PASS: Prediction generated → 0.7191228416019138
PASS: Interval calculation correct
PASS: Classification logic correct
PASS: Feature importance computed
PASS: Full system pipeline works


### Integration Testing

In [31]:
# Integration Test 1: End-to-End Pipeline Validation
def integration_test_full_pipeline():
    sample = X_test.iloc[[1]]
    listing_price = y_test.iloc[1] * 1.05

    result = housebroker_system(
        sample,
        model,
        scaler,
        feature_columns,
        residuals,
        listing_price
    )

    assert isinstance(result, dict)
    assert "Predicted Price" in result
    assert "Price Range" in result
    assert "Classification" in result
    assert "Top Features" in result

    print("PASS: Full pipeline integration successful")

In [32]:
# Integration Test 2: Data Flow Consistency
def integration_test_data_flow():
    sample = X_test.iloc[[2]]

    processed = preprocess_input(sample, scaler, feature_columns)
    pred = predict_price(model, processed)
    lower, upper = compute_interval(pred, residuals)

    assert processed.shape[1] == len(feature_columns)
    assert lower < pred < upper

    print("PASS: Data flow across components is consistent")

In [33]:
# Integration Test 3: Classification with Pipeline Output
def integration_test_classification_flow():
    sample = X_test.iloc[[3]]
    listing_price = y_test.iloc[3] * 1.2

    processed = preprocess_input(sample, scaler, feature_columns)
    pred = predict_price(model, processed)
    classification = classify_price(pred, listing_price)

    assert classification in ["Overpriced", "Underpriced", "Fairly Priced"]

    print("PASS: Classification integrated correctly with prediction")

In [34]:
# Integration Test 4: Feature Importance Integration
def integration_test_feature_importance():
    features = feature_importance(model, feature_columns)

    assert len(features) > 0
    assert isinstance(features[0][0], str)

    print("PASS: Feature importance integrated with model")

In [35]:
# Run All
def run_integration_tests():
    integration_test_full_pipeline()
    integration_test_data_flow()
    integration_test_classification_flow()
    integration_test_feature_importance()

run_integration_tests()

PASS: Full pipeline integration successful
PASS: Data flow across components is consistent
PASS: Classification integrated correctly with prediction
PASS: Feature importance integrated with model
